# images.cv Dataset Starter

**Download, inspect and visualize a computer vision dataset from [images.cv](https://images.cv).**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/images-cv/images.cv/blob/main/notebooks/images_cv_starter.ipynb)

This notebook is a first-time-visitor-friendly walkthrough that:

1. Downloads a public dataset from images.cv using the real, anonymous API.
2. Verifies and safely extracts the package.
3. Explores its metadata, images, and class distribution.
4. Visualizes bounding boxes / masks / COCO / YOLO annotations **when the
   package actually includes them** (most public datasets today are
   images-only — see `docs/dataset-package-format.md` in the repo).
5. Optionally builds a minimal PyTorch `Dataset`/`DataLoader`.

Run cells top to bottom. You don't need to edit any code — only the form
fields in **Dataset configuration** below. It works on the free CPU runtime.

> Dataset licenses vary. Review the license and attribution requirements
> shown for each dataset on [images.cv](https://images.cv) before using or
> redistributing it. This notebook's code is separate from any individual
> dataset's license.


## 1. Dataset configuration

Pick a dataset by its **slug** (the short key images.cv uses internally,
e.g. `pug`, `buoy`, `golden_pheasant`). You can find slugs by browsing
[images.cv](https://images.cv) — the slug is the last part of a dataset's
URL — or via the site's search.

There is currently no way to select a dataset **version** — every download
builds fresh from whatever images currently belong to that category (see
`docs/dataset-package-format.md`), so there is no `DATASET_VERSION` field
here.


In [ ]:
# @markdown ### Dataset
DATASET_SLUG = "buoy"  # @param {type:"string"}

# @markdown ### Download options (mirrors the real images.cv download form)
OUTPUT_SIZE = "size_128"  # @param ["size_16", "size_32", "size_64", "size_128", "size_256", "nochange"]
COLOR_MODE = "color"  # @param ["color", "gray"]
SPLIT_TRAIN_PCT = 70  # @param {type:"slider", min:0, max:100, step:5}
SPLIT_VAL_PCT = 15  # @param {type:"slider", min:0, max:100, step:5}
# test % is whatever remains after train + val

# @markdown ### Behavior
FORCE_DOWNLOAD = False  # @param {type:"boolean"}
DOWNLOAD_TIMEOUT_SECONDS = 600  # @param {type:"integer"}

# @markdown ### Exploration
SAMPLE_COUNT = 12  # @param {type:"slider", min:4, max:48, step:4}
RANDOM_SEED = 42  # @param {type:"integer"}

# @markdown ### Optional sections (off by default -- see sections 15 and 16)
RUN_PYTORCH_EXAMPLE = False  # @param {type:"boolean"}
MOUNT_GOOGLE_DRIVE = False  # @param {type:"boolean"}

SPLIT_TEST_PCT = max(0, 100 - SPLIT_TRAIN_PCT - SPLIT_VAL_PCT)
SPLIT_SET = [SPLIT_TRAIN_PCT / 100, SPLIT_VAL_PCT / 100, SPLIT_TEST_PCT / 100]

assert DATASET_SLUG.strip(), "Set DATASET_SLUG to a dataset key, e.g. 'pug'."
print(f"Configured dataset: {DATASET_SLUG!r}")
print(f"Split (train/val/test): {SPLIT_TRAIN_PCT}% / {SPLIT_VAL_PCT}% / {SPLIT_TEST_PCT}%")


## 2. Environment setup

Installs only what Colab doesn't already provide. `pycocotools` is optional
and only used for COCO RLE segmentation masks, if a dataset ever includes
them.


In [ ]:
import importlib
import subprocess
import sys

REQUIRED = ["requests", "pandas", "PIL", "matplotlib", "tqdm", "yaml"]
MODULE_TO_PACKAGE = {"PIL": "Pillow", "yaml": "PyYAML"}

missing = []
for module_name in REQUIRED:
    try:
        importlib.import_module(module_name)
    except ImportError:
        missing.append(MODULE_TO_PACKAGE.get(module_name, module_name))

if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("All required packages are already available.")

# pycocotools is optional -- only needed for COCO RLE masks.
try:
    import pycocotools  # noqa: F401
    HAS_PYCOCOTOOLS = True
except ImportError:
    HAS_PYCOCOTOOLS = False
    print("pycocotools not available -- COCO RLE masks will be skipped (polygon masks still work).")

import io
import json
import random
import secrets
import time
import zipfile
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import pandas as pd
import requests
import yaml
from PIL import Image, ImageFile
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True  # don't crash the whole notebook on one truncated image

random.seed(RANDOM_SEED)

API_BASE_URL = "https://api.images.cv"

# The production API sits behind Cloudflare. Requests without a realistic
# browser-like User-Agent (and ideally Origin/Referer) are blocked with an
# HTML "Attention Required" 403 page even though the request itself is
# well-formed -- verified empirically while building this notebook. See
# docs/notebook-usage.md's Troubleshooting section.
REQUEST_HEADERS = {
    "Content-Type": "application/json",
    "Accept": "application/json",
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36"
    ),
    "Origin": "https://images.cv",
    "Referer": "https://images.cv/",
}

WORK_DIR = Path("images_cv_workspace") / DATASET_SLUG
WORK_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = WORK_DIR / f"{DATASET_SLUG}.zip"
EXTRACT_DIR = WORK_DIR / "package"

print(f"Working directory: {WORK_DIR.resolve()}")


## 3. Retrieve metadata

Calls the real `/get_items_image_category` endpoint. This is the same
call the images.cv dataset page itself makes for its preview panel.

**Note:** this endpoint does not return a license, description, or
maintainer field -- images.cv does not currently expose those over a
public API, only on the dataset's web page. Machine-generated annotation
status and review status are shown here only when the package itself
contains that information (see §7-8) -- there is no metadata field for
them today.


In [ ]:
def api_post(path, body, timeout=45, retries=2):
    # The backend runs on Cloud Functions and can take 15-25s to respond on
    # a cold start -- observed directly while building this notebook. 45s
    # gives margin above that without hanging forever on a dead backend.
    # Retries only cover network-level failures (timeouts, connection resets),
    # not application responses -- a real error response is returned as-is.
    resp = None
    for attempt in range(retries + 1):
        try:
            resp = requests.post(f"{API_BASE_URL}{path}", json=body, headers=REQUEST_HEADERS, timeout=timeout)
            break
        except requests.exceptions.RequestException as exc:
            if attempt == retries:
                raise RuntimeError(f"Network error calling {path}: {exc}") from exc
            time.sleep(2 * (attempt + 1))  # 2s, 4s backoff

    content_type = resp.headers.get("Content-Type", "")
    if "application/json" not in content_type:
        snippet = resp.text[:200].replace("\n", " ")
        raise RuntimeError(
            f"{path} returned non-JSON ({resp.status_code}, {content_type}). "
            f"This usually means the request was blocked before reaching the "
            f"app (e.g. a Cloudflare challenge page). First bytes: {snippet!r}"
        )
    data = resp.json()
    if resp.status_code != 200 or data.get("isError"):
        raise RuntimeError(f"{path} failed ({resp.status_code}): {data}")
    return data


metadata_response = api_post("/get_items_image_category", {"category": DATASET_SLUG, "start_after": 0})
meta_preview = metadata_response["data"]

metadata_rows = [
    ("Dataset slug", DATASET_SLUG),
    ("Image count", meta_preview.get("sumCounter")),
    ("Categories", ", ".join(meta_preview.get("categories") or []) or "-"),
    ("Main category", ", ".join(meta_preview.get("mainCategory") or []) or "-"),
    ("Data source", ", ".join(meta_preview.get("dataSource") or []) or "-"),
    ("Premium dataset", meta_preview.get("is_premium")),
    ("License", "Not available via API -- check the dataset page on images.cv"),
    ("Machine-generated annotations", "Unknown -- not exposed by this endpoint; see package inspection below"),
]
display(pd.DataFrame(metadata_rows, columns=["Field", "Value"]))

if not meta_preview.get("sumCounter"):
    print(f"\nWarning: '{DATASET_SLUG}' returned 0 images. Double-check the slug "
          f"matches a real images.cv dataset key.")


## 4. Download dataset

Submits a packaging job (`/download_images`), polls `/get_link` until it's
ready, then streams the ZIP with a progress bar. Reuses an existing
completed download unless `FORCE_DOWNLOAD` is enabled above.


In [ ]:
def submit_download_job(slug, output_size, color_type, split_set):
    link_id = secrets.token_hex(20)
    body = {
        "output_size": output_size,
        "color_type": color_type,
        "images": [],
        "categories": [slug],
        "augmentations": [],
        "split_set": split_set,
        "link_id": link_id,
        "request_type": "CALL_FROM_SEARCH",
        "user_email": [],
        "user_id": "",
        "augmentation_percent": 30,
    }
    result = api_post("/download_images", body)
    returned_id = result.get("data", {}).get("link_id")
    if not returned_id:
        raise RuntimeError(f"/download_images did not return a link_id: {result}")
    return returned_id


def wait_for_download_url(link_id, timeout_seconds=600, poll_interval=3.0):
    deadline = time.monotonic() + timeout_seconds
    with tqdm(total=timeout_seconds, unit="s", desc="Waiting for package") as bar:
        while time.monotonic() < deadline:
            result = api_post("/get_link", {"link_id": link_id})
            public_url = result.get("data", {}).get("public_url") or ""
            if public_url:
                bar.n = bar.total
                bar.refresh()
                return public_url
            time.sleep(poll_interval)
            bar.update(poll_interval)
    raise RuntimeError(
        f"Timed out after {timeout_seconds}s waiting for the package to be built. "
        f"Try increasing DOWNLOAD_TIMEOUT_SECONDS for large or first-time categories."
    )


def stream_download(url, dest_zip):
    resp = requests.get(url, stream=True, timeout=60)
    content_type = resp.headers.get("Content-Type", "")
    if resp.status_code != 200 or "html" in content_type.lower():
        snippet = resp.text[:200].replace("\n", " ") if not resp.raw.closed else ""
        raise RuntimeError(
            f"Package download returned {resp.status_code} ({content_type}), expected a ZIP. "
            f"First bytes: {snippet!r}"
        )

    total = int(resp.headers.get("Content-Length", 0))
    dest_zip.parent.mkdir(parents=True, exist_ok=True)
    written = 0
    with open(dest_zip, "wb") as fh, tqdm(total=total or None, unit="B", unit_scale=True, desc="Downloading") as bar:
        for chunk in resp.iter_content(chunk_size=1024 * 1024):
            if not chunk:
                continue
            fh.write(chunk)
            written += len(chunk)
            bar.update(len(chunk))

    with open(dest_zip, "rb") as fh:
        magic = fh.read(4)
    if magic[:2] != b"PK":
        dest_zip.unlink(missing_ok=True)
        raise RuntimeError(
            "Downloaded file is not a valid ZIP (missing 'PK' signature) -- "
            "the response was likely an error page rather than the package."
        )
    return written


if ZIP_PATH.exists() and not FORCE_DOWNLOAD:
    print(f"{ZIP_PATH} already exists -- reusing it (enable FORCE_DOWNLOAD above to re-fetch).")
else:
    print(f"Submitting packaging job for '{DATASET_SLUG}' "
          f"(size={OUTPUT_SIZE}, color={COLOR_MODE}, split={SPLIT_SET})...")
    job_link_id = submit_download_job(DATASET_SLUG, OUTPUT_SIZE, COLOR_MODE, SPLIT_SET)
    print(f"Job accepted (link_id={job_link_id}). This can take from seconds to several minutes.")
    ready_url = wait_for_download_url(job_link_id, timeout_seconds=DOWNLOAD_TIMEOUT_SECONDS)
    bytes_written = stream_download(ready_url, ZIP_PATH)
    print(f"Downloaded {bytes_written / (1024 * 1024):.1f} MB -> {ZIP_PATH}")

print("Note: images.cv does not provide a checksum for this package -- "
      "integrity verification beyond the ZIP signature ('PK' header) is unavailable.")


## 5. Verify and extract package

Extracts the ZIP defensively: rejects any member path that would escape
the target directory (zip-slip / path traversal), and never loads the
whole archive into RAM.


In [ ]:
def safe_extract(zip_path, dest_dir):
    dest_dir.mkdir(parents=True, exist_ok=True)
    resolved_dest = dest_dir.resolve()
    with zipfile.ZipFile(zip_path) as zf:
        for member in zf.infolist():
            member_path = (dest_dir / member.filename).resolve()
            if not str(member_path).startswith(str(resolved_dest)):
                raise RuntimeError(f"Refusing to extract unsafe path: {member.filename!r}")
        zf.extractall(dest_dir)
        return sum(m.file_size for m in zf.infolist())


archive_size_mb = ZIP_PATH.stat().st_size / (1024 * 1024)
print(f"Archive size: {archive_size_mb:.1f} MB")

if EXTRACT_DIR.exists() and FORCE_DOWNLOAD:
    import shutil
    shutil.rmtree(EXTRACT_DIR)

if not EXTRACT_DIR.exists() or not any(EXTRACT_DIR.iterdir()):
    extracted_bytes = safe_extract(ZIP_PATH, EXTRACT_DIR)
    print(f"Extracted {extracted_bytes / (1024 * 1024):.1f} MB -> {EXTRACT_DIR}")
else:
    print(f"{EXTRACT_DIR} already populated -- skipping re-extraction.")

# The package is wrapped in a single top-level folder (e.g. images.cv_<id>/) -- unwrap it.
_roots = [p for p in EXTRACT_DIR.iterdir() if p.is_dir()]
PACKAGE_ROOT = _roots[0] if len(_roots) == 1 and not (EXTRACT_DIR / "meta.json").exists() else EXTRACT_DIR
print(f"Package root: {PACKAGE_ROOT}")


## 6. Inspect dataset structure

Detects which parts of the package actually exist. **Nothing below fails
the notebook if a capability is missing** -- most public datasets today
are images-only. See `docs/dataset-package-format.md` for the full current
vs. planned format breakdown.


In [ ]:
DATA_DIR = PACKAGE_ROOT / "data"
IMAGES_SUBDIR = DATA_DIR / "images" if (DATA_DIR / "images").is_dir() else DATA_DIR

CAPABILITIES = {
    "Images": any(DATA_DIR.rglob("*.jpg")) or any(DATA_DIR.rglob("*.jpeg")) or any(DATA_DIR.rglob("*.png")),
    "Labels": any((DATA_DIR / s).is_dir() and any((DATA_DIR / s).iterdir())
                  for s in ("train", "val", "test") if (DATA_DIR / s).is_dir()),
    "Bounding boxes": (PACKAGE_ROOT / "bboxes").is_dir(),
    "Masks": (PACKAGE_ROOT / "masks").is_dir(),
    "COCO": (PACKAGE_ROOT / "coco").is_dir(),
    "YOLO detection": (PACKAGE_ROOT / "yolo" / "detection").is_dir() or (PACKAGE_ROOT / "yolo" / "labels").is_dir(),
    "YOLO segmentation": (PACKAGE_ROOT / "yolo" / "segmentation").is_dir(),
}

display(pd.DataFrame(
    [(k, "Yes" if v else "No") for k, v in CAPABILITIES.items()],
    columns=["Capability", "Available"],
))

HAS_INDEX_CSV = (PACKAGE_ROOT / "index.csv").is_file()
print(f"index.csv present: {HAS_INDEX_CSV}")


## 7. Explore package metadata (`meta.json`)


In [ ]:
meta_json_path = PACKAGE_ROOT / "meta.json"
if meta_json_path.exists():
    package_meta = json.loads(meta_json_path.read_text(encoding="utf-8"))
    print(json.dumps(package_meta, indent=2))
else:
    package_meta = {}
    print("This package does not include a meta.json file.")


## 8. Build an image index

Scans `data/{train,val,test}/<label>/` (falling back to a flat `data/` or
`data/images/` layout if that's what the package has) into a single table
used by every section below. Handles missing/corrupted/unsupported files
without stopping the notebook.


In [ ]:
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff"}


def build_image_index(data_dir):
    rows = []
    corrupted = 0
    splits_present = [s for s in ("train", "val", "test") if (data_dir / s).is_dir()]
    search_roots = [(s, data_dir / s) for s in splits_present] or [("all", data_dir)]

    for split_name, split_dir in search_roots:
        for path in split_dir.rglob("*"):
            if not path.is_file() or path.suffix.lower() not in SUPPORTED_EXTENSIONS:
                continue
            label = path.parent.name if path.parent != split_dir else "unlabeled"
            try:
                with Image.open(path) as img:
                    width, height = img.size
            except Exception:
                corrupted += 1
                continue
            rows.append({
                "path": str(path),
                "filename": path.name,
                "label": label,
                "split": split_name,
                "width": width,
                "height": height,
            })

    if corrupted:
        print(f"Skipped {corrupted} unreadable/corrupted image file(s).")
    return pd.DataFrame(rows)


image_index = build_image_index(DATA_DIR)
print(f"Indexed {len(image_index)} images across {image_index['label'].nunique() if len(image_index) else 0} labels.")
image_index.head()


## 9. Display random images


In [ ]:
def show_random_images(index_df, sample_count, seed):
    if index_df.empty:
        print("No images were indexed -- nothing to display.")
        return
    sample_count = min(sample_count, len(index_df), 48)  # hard cap to avoid accidental memory blowups
    sample = index_df.sample(n=sample_count, random_state=seed)

    cols = min(4, sample_count)
    rows = (sample_count + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = axes.flatten() if sample_count > 1 else [axes]

    for ax, (_, row) in zip(axes, sample.iterrows()):
        try:
            with Image.open(row["path"]) as img:
                ax.imshow(img.convert("RGB"))
        except Exception:
            ax.text(0.5, 0.5, "unreadable", ha="center", va="center")
        ax.set_title(f"{row['label']}\n{row['width']}x{row['height']} ({row['split']})", fontsize=9)
        ax.axis("off")

    for ax in axes[len(sample):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


show_random_images(image_index, SAMPLE_COUNT, RANDOM_SEED)


## 10. Class and split distribution


In [ ]:
if image_index.empty:
    print("No images were indexed -- skipping distribution plots.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    label_counts = image_index["label"].value_counts()
    label_counts.plot(kind="bar", ax=axes[0])
    axes[0].set_title("Images per label")
    axes[0].set_ylabel("count")

    split_counts = image_index["split"].value_counts()
    split_counts.plot(kind="bar", ax=axes[1], color="orange")
    axes[1].set_title("Images per split")

    plt.tight_layout()
    plt.show()

    print(f"Image width range: {image_index['width'].min()}-{image_index['width'].max()}px")
    print(f"Image height range: {image_index['height'].min()}-{image_index['height'].max()}px")


## 11. Bounding box visualization

Supports canonical pixel-space bbox JSON, COCO `[x, y, w, h]`, and YOLO
normalized `[cx, cy, w, h]` -- all converted to a single canonical
original-image-pixel XYXY representation before drawing, so the drawing
code never needs to know which format it came from.

Public images.cv datasets do not currently include bounding boxes (see
`docs/dataset-package-format.md`) -- this section runs safely and shows a
message instead of failing when `bboxes/` isn't present.


In [ ]:
def coco_xywh_to_xyxy(box, img_w, img_h):
    x, y, w, h = box
    return _clip_xyxy([x, y, x + w, y + h], img_w, img_h)


def yolo_cxcywh_to_xyxy(box, img_w, img_h):
    cx, cy, w, h = box
    x1 = (cx - w / 2) * img_w
    y1 = (cy - h / 2) * img_h
    x2 = (cx + w / 2) * img_w
    y2 = (cy + h / 2) * img_h
    return _clip_xyxy([x1, y1, x2, y2], img_w, img_h)


def _clip_xyxy(box, img_w, img_h):
    x1, y1, x2, y2 = box
    if not all(map(_is_finite, box)) or x2 <= x1 or y2 <= y1:
        return None
    x1, y1 = max(0.0, x1), max(0.0, y1)
    x2, y2 = min(float(img_w), x2), min(float(img_h), y2)
    if x2 <= x1 or y2 <= y1:
        return None
    return [x1, y1, x2, y2]


def _is_finite(value):
    try:
        return value == value and abs(value) != float("inf")
    except TypeError:
        return False


DETERMINISTIC_COLORS = plt.get_cmap("tab20").colors


def color_for_label(label):
    return DETERMINISTIC_COLORS[hash(label) % len(DETERMINISTIC_COLORS)]


def draw_boxes(image_path, boxes):
    """boxes: list of dicts with canonical xyxy 'box', 'label', optional 'score', 'instance', 'machine_generated'."""
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        fig, ax = plt.subplots(1, figsize=(6, 6))
        ax.imshow(img)
        invalid = 0
        for i, ann in enumerate(boxes):
            box = ann.get("box")
            if box is None:
                invalid += 1
                continue
            x1, y1, x2, y2 = box
            color = color_for_label(ann.get("label", "?"))
            rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor=color, facecolor="none")
            ax.add_patch(rect)
            label_text = str(ann.get("label", "?"))
            if ann.get("score") is not None:
                label_text += f" {ann['score']:.2f}"
            if ann.get("machine_generated"):
                label_text += " [auto]"
            ax.text(x1, max(0, y1 - 4), label_text, color="white", fontsize=8,
                     bbox=dict(facecolor=color, alpha=0.8, pad=1))
        ax.axis("off")
        plt.tight_layout()
        plt.show()
        if invalid:
            print(f"Skipped {invalid} invalid annotation(s) (non-finite or degenerate box).")


if not CAPABILITIES["Bounding boxes"]:
    print("This dataset package does not include bounding boxes.")
else:
    bbox_dir = PACKAGE_ROOT / "bboxes"
    sample_row = image_index.sample(n=1, random_state=RANDOM_SEED).iloc[0]
    bbox_file = bbox_dir / f"{Path(sample_row['filename']).stem}.json"
    if bbox_file.exists():
        raw_boxes = json.loads(bbox_file.read_text(encoding="utf-8"))
        canonical = [
            {"box": _clip_xyxy(b["box"], sample_row["width"], sample_row["height"]),
             "label": b.get("label"), "score": b.get("score"),
             "machine_generated": b.get("machine_generated")}
            for b in raw_boxes
        ]
        draw_boxes(sample_row["path"], canonical)
    else:
        print(f"No bbox annotation file found for sample image {sample_row['filename']!r}.")


## 12. Mask visualization

Supports binary PNG masks, COCO polygon/RLE segmentation (RLE requires
`pycocotools`), and YOLO segmentation polygons -- overlaid semi-transparently
with distinct colors per instance.


In [ ]:
import numpy as np


def overlay_mask(image_path, mask_array, alpha=0.5):
    with Image.open(image_path) as img:
        img = np.array(img.convert("RGB"))
    colored_mask = np.zeros_like(img)
    instance_ids = [i for i in np.unique(mask_array) if i != 0]
    for idx, inst_id in enumerate(instance_ids):
        color = np.array(DETERMINISTIC_COLORS[idx % len(DETERMINISTIC_COLORS)]) * 255
        colored_mask[mask_array == inst_id] = color

    overlay = np.where(colored_mask.sum(axis=-1, keepdims=True) > 0,
                        (img * (1 - alpha) + colored_mask * alpha).astype(np.uint8), img)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img); axes[0].set_title("Original"); axes[0].axis("off")
    axes[1].imshow(overlay); axes[1].set_title("Overlay"); axes[1].axis("off")
    axes[2].imshow(mask_array, cmap="tab20"); axes[2].set_title("Mask only"); axes[2].axis("off")
    plt.tight_layout()
    plt.show()
    print(f"{len(instance_ids)} mask instance(s) found.")


if not CAPABILITIES["Masks"]:
    print("This dataset package does not include segmentation masks.")
else:
    masks_dir = PACKAGE_ROOT / "masks"
    sample_row = image_index.sample(n=1, random_state=RANDOM_SEED).iloc[0]
    mask_file = masks_dir / f"{Path(sample_row['filename']).stem}.png"
    if mask_file.exists():
        with Image.open(mask_file) as m:
            overlay_mask(sample_row["path"], np.array(m))
    else:
        print(f"No PNG mask found for sample image {sample_row['filename']!r} "
              f"(this dataset's masks may only be available as COCO/YOLO polygons -- see below).")


## 13. Inspect COCO annotations

Looks for the canonical `coco/instances.json` path first. This section is
skipped cleanly if the package has no `coco/` folder.


In [ ]:
if not CAPABILITIES["COCO"]:
    print("This dataset package does not include COCO annotations.")
else:
    coco_dir = PACKAGE_ROOT / "coco"
    coco_path = coco_dir / "instances.json"
    if not coco_path.exists():
        candidates = list(coco_dir.glob("*.json"))
        coco_path = candidates[0] if candidates else None

    if coco_path is None:
        print("coco/ exists but no annotation JSON file was found inside it.")
    else:
        coco_data = json.loads(coco_path.read_text(encoding="utf-8"))
        images_by_id = {im["id"]: im for im in coco_data.get("images", [])}
        categories_by_id = {c["id"]: c["name"] for c in coco_data.get("categories", [])}
        annotations = coco_data.get("annotations", [])

        missing_refs = sum(1 for a in annotations if a.get("image_id") not in images_by_id)
        print(f"COCO file: {len(images_by_id)} images, {len(categories_by_id)} categories, "
              f"{len(annotations)} annotations ({missing_refs} with an unresolved image_id).")

        if annotations:
            sample_ann = annotations[0]
            sample_image = images_by_id.get(sample_ann.get("image_id"))
            if sample_image:
                image_path = PACKAGE_ROOT / "data" / "images" / sample_image["file_name"]
                if not image_path.exists():
                    matches = list(DATA_DIR.rglob(sample_image["file_name"]))
                    image_path = matches[0] if matches else None
                if image_path and image_path.exists():
                    boxes_for_image = [
                        {"box": coco_xywh_to_xyxy(a["bbox"], sample_image["width"], sample_image["height"]),
                         "label": categories_by_id.get(a.get("category_id"), "?")}
                        for a in annotations if a.get("image_id") == sample_image["id"] and a.get("bbox")
                    ]
                    if boxes_for_image:
                        draw_boxes(image_path, boxes_for_image)


## 14. Inspect YOLO annotations

Supports both the proposed canonical layout (`yolo/data.yaml` +
`yolo/detection/labels/` or `yolo/segmentation/labels/`) and a flatter
`yolo/labels/` layout if that's what a given package uses -- detection and
segmentation label files are never mixed together.


In [ ]:
if not CAPABILITIES["YOLO detection"] and not CAPABILITIES["YOLO segmentation"]:
    print("This dataset package does not include YOLO annotations.")
else:
    yolo_dir = PACKAGE_ROOT / "yolo"
    data_yaml_path = yolo_dir / "data.yaml"
    class_names = []
    if data_yaml_path.exists():
        yolo_config = yaml.safe_load(data_yaml_path.read_text(encoding="utf-8"))
        class_names = yolo_config.get("names", [])
        print(f"data.yaml: {len(class_names)} classes -> {class_names}")
    else:
        print("No data.yaml found -- class names will show as numeric ids.")

    labels_dir = (yolo_dir / "detection" / "labels") if (yolo_dir / "detection" / "labels").is_dir() else (yolo_dir / "labels")
    is_segmentation = (yolo_dir / "segmentation" / "labels").is_dir() and not labels_dir.exists()
    if is_segmentation:
        labels_dir = yolo_dir / "segmentation" / "labels"

    label_files = list(labels_dir.glob("*.txt")) if labels_dir.is_dir() else []
    print(f"Found {len(label_files)} YOLO label file(s) in {labels_dir} "
          f"({'segmentation' if is_segmentation else 'detection'}).")

    if label_files and not is_segmentation:
        sample_label_file = label_files[0]
        matches = list(DATA_DIR.rglob(f"{sample_label_file.stem}.*"))
        image_matches = [m for m in matches if m.suffix.lower() in SUPPORTED_EXTENSIONS]
        if image_matches:
            image_path = image_matches[0]
            with Image.open(image_path) as im:
                img_w, img_h = im.size
            boxes = []
            for line in sample_label_file.read_text().splitlines():
                parts = line.split()
                if len(parts) != 5:
                    continue
                class_id, cx, cy, w, h = int(parts[0]), *map(float, parts[1:])
                if not all(0.0 <= v <= 1.0 for v in (cx, cy, w, h)):
                    continue
                label = class_names[class_id] if class_id < len(class_names) else str(class_id)
                boxes.append({"box": yolo_cxcywh_to_xyxy([cx, cy, w, h], img_w, img_h), "label": label})
            if boxes:
                draw_boxes(image_path, boxes)


## 15. Minimal PyTorch `Dataset` / `DataLoader`

A simple classification-only example that reads from the image index built
above. Detection and segmentation training need task-specific collate
functions and annotation handling, which are out of scope for this starter
notebook. Does not require a GPU and does not download any pretrained
weights.


In [ ]:
if not RUN_PYTORCH_EXAMPLE:
    print("RUN_PYTORCH_EXAMPLE is False -- skipping (enable it in Dataset configuration above to run this section).")
elif image_index.empty:
    print("No images were indexed -- skipping PyTorch example.")
else:
    import torch
    from torch.utils.data import DataLoader, Dataset
    from torchvision import transforms

    class ImagesCvDataset(Dataset):
        def __init__(self, df, transform):
            self.df = df.reset_index(drop=True)
            self.labels = sorted(df["label"].unique())
            self.label_to_idx = {label: i for i, label in enumerate(self.labels)}
            self.transform = transform

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            with Image.open(row["path"]) as img:
                image = self.transform(img.convert("RGB"))
            return image, self.label_to_idx[row["label"]]

    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
    ])
    torch_dataset = ImagesCvDataset(image_index, transform)
    torch_loader = DataLoader(torch_dataset, batch_size=min(8, len(torch_dataset)), shuffle=True)

    images_batch, labels_batch = next(iter(torch_loader))
    print(f"Batch shape: {images_batch.shape}, labels: {labels_batch.tolist()}")
    print(f"Classes: {torch_dataset.labels}")

    fig, axes = plt.subplots(1, len(images_batch), figsize=(2.5 * len(images_batch), 2.5))
    axes = axes if len(images_batch) > 1 else [axes]
    for ax, img_tensor, label_idx in zip(axes, images_batch, labels_batch):
        ax.imshow(img_tensor.permute(1, 2, 0).numpy())
        ax.set_title(torch_dataset.labels[label_idx], fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


## 16. Optional: export to Google Drive

Only mounts Drive if `MOUNT_GOOGLE_DRIVE` is enabled above. Nothing is
written to Drive by default.


In [ ]:
if not MOUNT_GOOGLE_DRIVE:
    print("MOUNT_GOOGLE_DRIVE is False -- skipping (enable it in Dataset configuration above to run this section).")
else:
    from google.colab import drive
    drive.mount("/content/drive")

    export_dir = Path("/content/drive/MyDrive") / "images_cv_exports" / DATASET_SLUG
    export_dir.mkdir(parents=True, exist_ok=True)

    import shutil
    shutil.copy2(ZIP_PATH, export_dir / ZIP_PATH.name)
    if not image_index.empty:
        image_index.to_csv(export_dir / "image_index.csv", index=False)
    (export_dir / "capabilities.json").write_text(json.dumps(CAPABILITIES, indent=2), encoding="utf-8")

    print(f"Exported dataset ZIP, image index, and capability report to {export_dir}")


## 17. Troubleshooting and next steps

**"non-JSON" or Cloudflare-related errors** — the production API is behind
Cloudflare, which blocks requests that don't look like they came from a
browser. This notebook already sends browser-like headers; if you still hit
this, you may be running from a heavily rate-limited or already-flagged
network -- try again later or from a different network.

**A metadata or job-submit call is slow (10-25s) or times out** — the
backend runs on Cloud Functions, which can cold-start in that range;
this notebook already allows 45s per call. If it still times out, just
re-run the cell.

**Timed out waiting for the package** — large, first-time-requested
categories take longer to build. Increase `DOWNLOAD_TIMEOUT_SECONDS` in the
configuration cell and re-run from §4.

**0 images / dataset not found** — double check `DATASET_SLUG` against a
real images.cv dataset URL slug.

**A capability section printed "does not include..."** — that's expected
for most datasets today; see `docs/dataset-package-format.md` for exactly
what's currently included vs. planned.

**Corrupted/unreadable images** — silently skipped and counted; they don't
stop the notebook.

### Next steps

- Browse more datasets at [images.cv](https://images.cv).
- Read `docs/notebook-usage.md` for Colab-specific tips (free-tier limits,
  saving a copy, CPU vs GPU).
- Report notebook problems by opening an issue in the
  [images-cv/images.cv](https://github.com/images-cv/images.cv) repository.
